In [1]:
import os, json, csv, re
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
from typing import Any, Dict, List, Optional
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def process_video_jsons(root_dir, output_csv):
    video_data = {} # Groups results by video ID
    video_metadata = {} # Stores technical metadata per video
    
    for filename in os.listdir(root_dir):
        if filename.endswith(".json"):
            # Regex to extract video ID before the window pattern (_wXXX)
            match = re.match(r"(.*)_w\d+_\d+-\d+\.json", filename)
            if not match: continue
            video_base_name = match.group(1)
            file_path = os.path.join(root_dir, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    content = data.get("content", "").lower()
                    # Determine if current window shows abuse
                    current_result = "Yes" if "yes" in content else "No"
                    if video_base_name not in video_data:
                        video_data[video_base_name] = []
                        # Flag video as Normal if filename starts with 'normal'
                        is_norm_flag = "True" if video_base_name.startswith("normal") else "False"
                        video_metadata[video_base_name] = {
                            "is_normal": is_norm_flag,
                            "sample_fps": data.get("sample_fps"),
                            "window_seconds": data.get("window_seconds"),
                            "stride_ratio": data.get("stride_ratio"),
                            "start_frame_idx": data.get("start_frame_idx"),
                            "end_frame_idx": data.get("end_frame_idx"),
                            "num_frames": data.get("num_frames")
                        }
                    video_data[video_base_name].append(current_result)
            except Exception as e: print(f"Error reading {filename}: {e}")

    # Write aggregated data to CSV
    headers = ["video_name", "is_normal", "final_prediction", "sample_fps", "window_seconds", "stride_ratio", "start_frame_idx", "end_frame_idx", "num_frames"]
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        for v_name, results in video_data.items():
            # Rule: "Yes" if any window is "Yes". For Normal videos, "No" only if ALL are "No".
            final_pred = "Yes" if "Yes" in results else "No"
            row = {"video_name": v_name, "final_prediction": final_pred}
            row.update(video_metadata[v_name])
            writer.writerow(row)
    print(f"Stats saved to: {output_csv}")

In [ ]:
# Execution
target_folder = './anomaly_win40s_stride20pct_fps2.0/abuse'
process_video_jsons(target_folder, 'video_analysis_results.csv')